In [1]:
# ============================================================
# M1 — LOAD FINAL SCALED EI03 NETWORK (ENERGY ISLAND, FULL YEAR)
# ============================================================

import pypsa
import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# PATHS (HARD, EXPLICIT — NO KERNEL STATE)
# ------------------------------------------------------------
root = Path(r"C:\Users\franc\pypsa\pypsa-eur")

SCENARIO = "EI03_energy_island"

base_fn = (
    root
    / "results"
    / "networks"
    / f"{SCENARIO}.nc"
)

print("M1 loading network from:")
print(base_fn)

assert base_fn.exists(), f"Scaled EI03 network not found: {base_fn}"

# ------------------------------------------------------------
# LOAD NETWORK (FULL YEAR, 8760)
# ------------------------------------------------------------
n = pypsa.Network(base_fn)
print("Loaded:", base_fn.name)
print("Snapshots:", len(n.snapshots))

# ------------------------------------------------------------
# SAFETY — ENSURE CARRIER TABLE IS NETCDF / PYPSA SAFE
# ------------------------------------------------------------
if "color" in n.carriers.columns:
    n.carriers["color"] = n.carriers["color"].astype(str)

# Ensure all generator carriers exist in n.carriers
n.generators["carrier"] = n.generators["carrier"].astype(str)
missing_carriers = set(n.generators.carrier.unique()) - set(n.carriers.index)

if missing_carriers:
    print("INFO: Adding missing carriers to n.carriers:", sorted(missing_carriers))
    default_row = {c: 0 for c in n.carriers.columns}
    for mc in sorted(missing_carriers):
        n.carriers.loc[mc] = default_row

# ------------------------------------------------------------
# DISPATCH-ONLY (ABSOLUTE — NO EXPANSION)
# ------------------------------------------------------------
for df_name in ["generators", "links", "lines", "storage_units"]:
    if hasattr(n, df_name):
        df = getattr(n, df_name)
        if "p_nom_extendable" in df.columns:
            df["p_nom_extendable"] = False

if hasattr(n, "lines") and "s_nom_extendable" in n.lines.columns:
    n.lines["s_nom_extendable"] = False

print("✓ Expansion disabled (dispatch-only mode)")

# ------------------------------------------------------------
# STORAGE — NON-CYCLIC, DAILY ROLLING COMPATIBLE
# (Battery + any other storage)
# ------------------------------------------------------------
if hasattr(n, "storage_units") and not n.storage_units.empty:
    if "cyclic_state_of_charge" in n.storage_units.columns:
        n.storage_units["cyclic_state_of_charge"] = False
    if "state_of_charge_initial" not in n.storage_units.columns:
        n.storage_units["state_of_charge_initial"] = 0.0

if hasattr(n, "stores") and not n.stores.empty:
    if "cyclic_state_of_charge" in n.stores.columns:
        n.stores["cyclic_state_of_charge"] = False
    if "e_initial" not in n.stores.columns:
        n.stores["e_initial"] = 0.0

print("✓ Storage set to non-cyclic with explicit initial states")

# ------------------------------------------------------------
# LOAD SHEDDING — LAST RESORT (GLOBAL SAFETY)
# ------------------------------------------------------------
LS_COST = 10_000.0  # €/MWh

if "load_shedding" not in n.carriers.index:
    n.carriers.loc["load_shedding"] = {c: 0 for c in n.carriers.columns}

missing_ls = [
    b for b in n.buses.index
    if f"load_shed_{b}" not in n.generators.index
]

for b in missing_ls:
    n.add(
        "Generator",
        name=f"load_shed_{b}",
        bus=b,
        carrier="load_shedding",
        p_nom=1e6,
        p_min_pu=0.0,
        p_max_pu=1.0,
        marginal_cost=LS_COST,
    )

print("✓ Load shedding generators ensured:", len(missing_ls))

# ------------------------------------------------------------
# SNAPSHOT BACKUP (FOR M2 / M3 DAILY SLICING)
# ------------------------------------------------------------
FULL_SNAPSHOTS = n.snapshots.copy()

# ------------------------------------------------------------
# OUTPUT DIRECTORIES (EI03-SPECIFIC)
# ------------------------------------------------------------
out_root = root / "results" / "daily_runs_EI03"
by_day   = out_root / "by_day"
by_month = out_root / "by_month"

by_day.mkdir(parents=True, exist_ok=True)
by_month.mkdir(parents=True, exist_ok=True)

print("✓ Output directories ready:")
print(" ", out_root)

# ------------------------------------------------------------
# SOLVER
# ------------------------------------------------------------
SOLVER = "cbc"   # robust for daily rolling horizon
print("Solver set to:", SOLVER)

# ------------------------------------------------------------
# SANITY CHECK — ENERGY ISLAND CORE COMPONENTS
# ------------------------------------------------------------
assert "EI03_bus" in n.buses.index, "Energy Island bus missing"
assert (n.generators.carrier.str.contains("offwind")).any(), "Offshore wind missing"
assert (n.links.carrier == "DC").any(), "HVDC links missing"
assert (n.links.carrier == "electrolysis").any(), "Electrolyser missing"
assert (n.storage_units.carrier == "battery").any(), "Battery missing"

print("✓ Energy Island core components detected")

print("\nM1 READY — proceed to M2")


M1 loading network from:
C:\Users\franc\pypsa\pypsa-eur\results\networks\EI03_energy_island.nc


INFO:pypsa.io:Imported network EI03_energy_island.nc has buses, carriers, generators, lines, links, loads, storage_units, stores


Loaded: EI03_energy_island.nc
Snapshots: 8760
✓ Expansion disabled (dispatch-only mode)
✓ Storage set to non-cyclic with explicit initial states
✓ Load shedding generators ensured: 0
✓ Output directories ready:
  C:\Users\franc\pypsa\pypsa-eur\results\daily_runs_EI03
Solver set to: cbc
✓ Energy Island core components detected

M1 READY — proceed to M2


In [2]:
# ============================================================
# M2_EI03 — Helpers for one-day solve, metrics, and rolling states
# ENERGY ISLAND SCENARIO
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# CONSTANTS
# ------------------------------------------------------------

EI_BUS = "EI03_bus"
H2_BUS = "H2"
EI_COUNTRY = "EI"

MAIN_COUNTRIES = ["DK", "DE", "NL", "NO", "SE"]

# ------------------------------------------------------------
# Country mappings (EXCLUDING Energy Island)
# ------------------------------------------------------------

bus_country  = n.buses.country
gen_country  = n.generators.bus.map(bus_country)
load_country = n.loads.bus.map(bus_country)

# ------------------------------------------------------------
# Defensive: ensure all generator carriers exist in n.carriers
# ------------------------------------------------------------

n.generators["carrier"] = n.generators["carrier"].astype(str)
missing_carriers = set(n.generators.carrier.unique()) - set(n.carriers.index)

if missing_carriers:
    print("INFO (M2_EI03): adding missing carriers:", sorted(missing_carriers))
    default_row = {c: 0 for c in n.carriers.columns}
    for mc in missing_carriers:
        n.carriers.loc[mc] = default_row

# ------------------------------------------------------------
# Snapshot helpers
# ------------------------------------------------------------

def day_index(ts: pd.Timestamp) -> pd.DatetimeIndex:
    d = pd.Timestamp(ts).normalize()
    hours = pd.date_range(d, d + pd.Timedelta(hours=23), freq="h")
    return hours.intersection(FULL_SNAPSHOTS)

# ------------------------------------------------------------
# Rolling storage states (battery only)
# ------------------------------------------------------------

def roll_storage_initials_from_solution():
    if hasattr(n, "storage_units_t") and "state_of_charge" in n.storage_units_t:
        last_soc = (
            n.storage_units_t.state_of_charge.iloc[-1]
            .reindex(n.storage_units.index)
            .fillna(0.0)
        )
        if "state_of_charge_initial" not in n.storage_units.columns:
            n.storage_units["state_of_charge_initial"] = 0.0
        n.storage_units["state_of_charge_initial"] = last_soc

# ------------------------------------------------------------
# Metric extraction — EI03 LOGIC
# ------------------------------------------------------------

def extract_day_metrics(day_idx: pd.DatetimeIndex) -> dict:

    out = {}

    # -----------------
    # Core time series
    # -----------------

    loads_ts = n.loads_t.p_set.loc[day_idx]              # MW
    gens_ts  = n.generators_t.p.loc[day_idx]             # MW
    links_ts = n.links_t.p0.loc[day_idx] if hasattr(n, "links_t") else None

    # -----------------
    # DEMAND (excluding EI)
    # -----------------

    out["demand_EU_MW"] = float(loads_ts.sum(axis=1).mean())

    for c in MAIN_COUNTRIES:
        mask = (load_country == c)
        if mask.any():
            out[f"demand_{c}_MW"] = float(
                loads_ts.loc[:, mask].sum(axis=1).mean()
            )

    # -----------------
    # ELECTRIC GENERATION (EXCL. LOAD SHEDDING)
    # -----------------

    elec_gen_mask = ~n.generators.carrier.isin(["load_shedding", "hydrogen"])
    elec_gen_ts = gens_ts.loc[:, elec_gen_mask]

    out["generation_EU_MW"] = float(elec_gen_ts.sum(axis=1).mean())

    for c in MAIN_COUNTRIES:
        mask = (gen_country == c) & elec_gen_mask
        if mask.any():
            out[f"generation_{c}_MW"] = float(
                elec_gen_ts.loc[:, mask].sum(axis=1).mean()
            )

    # -----------------
    # ELECTRIC EXPORT FROM ISLAND (HVDC)
    # -----------------

    if links_ts is not None:
        hvdc_mask = n.links.bus0 == EI_BUS
        elec_export_ts = links_ts.loc[:, hvdc_mask].sum(axis=1)
        out["elec_export_MW"] = float(elec_export_ts.mean())
    else:
        out["elec_export_MW"] = 0.0

    # -----------------
    # ELECTRICITY TO H2 (ELECTROLYSER)
    # -----------------

    if links_ts is not None and "EI03_electrolyser" in n.links.index:
        elec_to_h2_ts = links_ts["EI03_electrolyser"]
        out["elec_to_h2_MW"] = float(elec_to_h2_ts.mean())
        out["h2_produced_MWh"] = float(elec_to_h2_ts.sum() * n.links.at["EI03_electrolyser", "efficiency"])
    else:
        out["elec_to_h2_MW"] = 0.0
        out["h2_produced_MWh"] = 0.0

    # -----------------
    # NET ELECTRIC BALANCE (EU)
    # -----------------

    out["net_elec_balance_EU_MW"] = (
        out["generation_EU_MW"]
        - out["demand_EU_MW"]
        - out["elec_to_h2_MW"]
    )

    # -----------------
    # PRICES
    # -----------------

    if hasattr(n, "buses_t") and "marginal_price" in n.buses_t:
        prices_ts = n.buses_t.marginal_price.loc[day_idx]

        out["price_EU_EUR_per_MWh"] = float(
            prices_ts.loc[:, bus_country != EI_COUNTRY].stack().mean()
        )

        for c in MAIN_COUNTRIES:
            mask = (bus_country == c)
            if mask.any():
                out[f"price_{c}_EUR_per_MWh"] = float(
                    prices_ts.loc[:, mask].mean(axis=1).mean()
                )

    # -----------------
    # LOAD SHEDDING
    # -----------------

    if (n.generators.carrier == "load_shedding").any():
        shed_ts = gens_ts.loc[:, n.generators.carrier == "load_shedding"]
        out["load_shed_MW"] = float(shed_ts.sum(axis=1).mean())
    else:
        out["load_shed_MW"] = 0.0

    return out

# ------------------------------------------------------------
# One-day solve wrapper
# ------------------------------------------------------------

def solve_one_day(
    d: pd.Timestamp,
    solver: str = None,
    threads: int = 4,
    logdir=None,
) -> dict:

    idx = day_index(d)
    if len(idx) == 0:
        return {"note": "no snapshots"}

    n.set_snapshots(idx)

    status, cond = n.optimize(
        solver_name=(solver or SOLVER),
        threads=threads,
        progress=False,
    )

    if status != "ok":
        return {"note": f"solve status {status} ({cond})"}

    metrics = extract_day_metrics(idx)
    roll_storage_initials_from_solution()
    return metrics

# ------------------------------------------------------------
# Smoke test
# ------------------------------------------------------------

d0 = pd.Timestamp("2013-01-01")
print("Testing EI03 day:", d0.date())
print(solve_one_day(d0, solver=SOLVER))
print("M2_EI03 ready.")


Testing EI03 day: 2013-01-01


INFO:linopy.model: Solve problem using Cbc solver
INFO:linopy.model:Solver options:
 - threads: 4
INFO:linopy.io: Writing time: 0.6s
INFO:linopy.solvers:Welcome to the CBC MILP Solver 
Version: 2.10.12 
Build Date: Aug  2 2025 

command line - cbc -printingOptions all -import C:\Users\franc\AppData\Local\Temp\linopy-problem-kqblmzh6.lp -threads 4 -solve -solu C:\Users\franc\AppData\Local\Temp\linopy-solve-559_zlva.sol (default strategy 1)
Option for printingOptions changed from normal to all
No match for threads - ? for list of commands
No match for 4 - ? for list of commands
Presolve 7258 (-62778) rows, 16737 (-17083) columns and 33903 (-85073) elements
Perturbing problem by 0.001% of 763.73472 - largest nonzero change 0.00086369541 ( 0.11070161%) - largest zero change 0.00086368374
0  Obj 80305408 Primal inf 16262335 (2061) Dual inf 7456.1598 (1)
220  Obj 45655411 Primal inf 18372332 (2035)
440  Obj 46692108 Primal inf 18882726 (2113)
660  Obj 46767961 Primal inf 23001459 (2182)
880 

{'demand_EU_MW': 322963.7659573934, 'demand_DK_MW': 3518.5416673024497, 'demand_DE_MW': 41525.18316332499, 'demand_NL_MW': 11271.708331425985, 'demand_NO_MW': 16243.166668256124, 'demand_SE_MW': 15982.95834604899, 'generation_EU_MW': 296857.7889794154, 'generation_DK_MW': 4471.804764554167, 'generation_DE_MW': 56736.522146250005, 'generation_NL_MW': 7942.780665416667, 'generation_NO_MW': 1400.184262075, 'generation_SE_MW': 12811.931077041665, 'elec_export_MW': 3650.0562491666665, 'elec_to_h2_MW': 1723.6655083333335, 'h2_produced_MWh': 28957.58054, 'net_elec_balance_EU_MW': -27829.64248631135, 'price_EU_EUR_per_MWh': 8.642501398611111, 'price_DK_EUR_per_MWh': 5.062240055555556, 'price_DE_EUR_per_MWh': 6.175060377777776, 'price_NL_EUR_per_MWh': 4.546726236111112, 'price_NO_EUR_per_MWh': 5.276068333333334, 'price_SE_EUR_per_MWh': 5.276068333333334, 'load_shed_MW': 714.4408208333333}
M2_EI03 ready.


In [14]:
# ============================================================
# FINAL M3_EI03 — FULL EXPORT (DAILY + HOURLY)
# ENERGY ISLAND SCENARIO (MULTI-VECTOR)
# BASELINE-ALIGNED + H2 EXTENSIONS
# ============================================================

import pandas as pd
import numpy as np
import pypsa
from pathlib import Path

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
SCENARIO = "EI03"
months = [f"2013-{m:02d}" for m in range(1, 13)]

MAIN_COUNTRIES = ["DK", "DE", "NL", "NO", "SE", "BE", "FR", "PL", "PT"]

root = Path(r"C:\Users\franc\pypsa\pypsa-eur")
base_fn  = root / "results" / "networks" / "EI03_energy_island.nc"
out_root = root / "results" / f"daily_runs_{SCENARIO}"
out_root.mkdir(parents=True, exist_ok=True)

SOLVER = "cbc"

print("=== FINAL M3_EI03 — COUNTRY-CORRECT (MULTI-VECTOR) ===")
print("Network:", base_fn)
assert base_fn.exists(), f"Network not found: {base_fn}"

# ------------------------------------------------------------
# SNAPSHOT HELPER
# ------------------------------------------------------------
def day_index(ts, full_snapshots):
    d = pd.Timestamp(ts).normalize()
    hours = pd.date_range(d, d + pd.Timedelta(hours=23), freq="h")
    return hours.intersection(full_snapshots)

# ------------------------------------------------------------
# MAIN LOOP
# ------------------------------------------------------------
for month in months:
    print(f"\n=== Month {month} ===")

    month_start = pd.Timestamp(f"{month}-01")
    month_end   = month_start + pd.offsets.MonthEnd(1)
    days = pd.date_range(month_start, month_end, freq="D")

    rows_daily  = []
    rows_hourly = []

    last_soc = None
    last_e   = None

    for d in days:
        # ----------------------------------------------------
        # LOAD NETWORK FRESH
        # ----------------------------------------------------
        n = pypsa.Network(base_fn)
        FULL_SNAPSHOTS = n.snapshots.copy()

        idx = day_index(d, FULL_SNAPSHOTS)
        if len(idx) == 0:
            continue

        print(f"→ {d.date()} ({len(idx)} h)")

        # ----------------------------------------------------
        # DISPATCH ONLY
        # ----------------------------------------------------
        for comp in ["generators", "links", "lines", "storage_units", "stores"]:
            if hasattr(n, comp):
                df = getattr(n, comp)
                if "p_nom_extendable" in df.columns:
                    df["p_nom_extendable"] = False

        if "s_nom_extendable" in n.lines.columns:
            n.lines["s_nom_extendable"] = False

        # ----------------------------------------------------
        # STORAGE (ROLLING)
        # ----------------------------------------------------
        if not n.storage_units.empty:
            n.storage_units["cyclic_state_of_charge"] = False
            if "state_of_charge_initial" not in n.storage_units.columns:
                n.storage_units["state_of_charge_initial"] = 0.0

        if not n.stores.empty:
            n.stores["cyclic_state_of_charge"] = False
            if "e_initial" not in n.stores.columns:
                n.stores["e_initial"] = 0.0

        if last_soc is not None and not n.storage_units.empty:
            n.storage_units["state_of_charge_initial"] = (
                last_soc.reindex(n.storage_units.index).fillna(0.0)
            )

        if last_e is not None and not n.stores.empty:
            n.stores["e_initial"] = (
                last_e.reindex(n.stores.index).fillna(0.0)
            )

        # ----------------------------------------------------
        # LOAD SHEDDING (SAFETY)
        # ----------------------------------------------------
        if "load_shedding" not in n.carriers.index:
            n.carriers.loc["load_shedding"] = {c: 0 for c in n.carriers.columns}

        for b in n.buses.index:
            g = f"load_shed_{b}"
            if g not in n.generators.index:
                n.add(
                    "Generator",
                    name=g,
                    bus=b,
                    carrier="load_shedding",
                    p_nom=1e6,
                    marginal_cost=10_000.0,
                )

        # ----------------------------------------------------
        # SOLVE DAY
        # ----------------------------------------------------
        n.set_snapshots(idx)
        status, cond = n.optimize(
            solver_name=SOLVER,
            threads=4,
            progress=False,
        )

        if status != "ok":
            rows_daily.append({"date": d.date(), "note": f"{status} ({cond})"})
            continue

        # ----------------------------------------------------
        # SAVE STORAGE STATE
        # ----------------------------------------------------
        if not n.storage_units.empty:
            last_soc = n.storage_units_t.state_of_charge.iloc[-1].copy()

        if not n.stores.empty:
            last_e = n.stores_t.e.iloc[-1].copy()

        # ----------------------------------------------------
        # BASE TIME SERIES
        # ----------------------------------------------------
        load_p  = n.loads_t.p_set.loc[idx]
        gen_p   = n.generators_t.p.loc[idx]
        prices  = n.buses_t.marginal_price.loc[idx]

        total_demand_MW     = load_p.sum(axis=1)
        total_generation_MW = gen_p.sum(axis=1)

        hourly_df = pd.DataFrame({
            "snapshot": idx,
            "date": idx.normalize(),
            "hour": idx.hour,
            "total_demand_MW": total_demand_MW.values,
            "total_generation_MW": total_generation_MW.values,
        })

        # ----------------------------------------------------
        # COUNTRY MAPS (BASELINE-COMPATIBLE)
        # ----------------------------------------------------
        bus_country = n.buses.country
        load_country = n.loads.bus.map(bus_country)
        gen_country  = n.generators.bus.map(bus_country)
        
        MAIN_COUNTRIES = ["DK", "DE", "NL", "NO", "SE", "BE", "FR", "PL", "PT"]
        
        # ----------------------------------------------------
        # DEMAND BY COUNTRY
        # ----------------------------------------------------
        for c in MAIN_COUNTRIES:
            mask = (load_country == c)
            if mask.any():
                hourly_df[f"demand_{c}_MW"] = load_p.loc[:, mask].sum(axis=1).values
            else:
                hourly_df[f"demand_{c}_MW"] = 0.0
        
        # ----------------------------------------------------
        # GENERATION BY COUNTRY
        # ----------------------------------------------------
        for c in MAIN_COUNTRIES:
            mask = (gen_country == c)
            if mask.any():
                hourly_df[f"generation_{c}_MW"] = gen_p.loc[:, mask].sum(axis=1).values
            else:
                hourly_df[f"generation_{c}_MW"] = 0.0
        
        # ----------------------------------------------------
        # PRICE BY COUNTRY (LOAD-WEIGHTED, LOAD BUSES ONLY)
        # ----------------------------------------------------
        prices_bus = n.buses_t.marginal_price.loc[idx]
        
        # load per bus (MW)
        load_by_bus = load_p.T.groupby(n.loads.bus).sum().T
        
        for c in MAIN_COUNTRIES:
            buses_c = bus_country[bus_country == c].index
            buses_c = buses_c.intersection(load_by_bus.columns)
        
            if len(buses_c) == 0:
                hourly_df[f"price_{c}_EUR_per_MWh"] = np.nan
                continue
        
            load_c = load_by_bus[buses_c]
            price_c = prices_bus[buses_c]
        
            denom = load_c.sum(axis=1).replace(0, np.nan)
            p_weighted = (price_c * load_c).sum(axis=1) / denom
        
            hourly_df[f"price_{c}_EUR_per_MWh"] = p_weighted.values


        # ----------------------------------------------------
        # LOAD SHEDDING (EU)
        # ----------------------------------------------------
        if (n.generators.carrier == "load_shedding").any():
            shed = gen_p.loc[:, n.generators.carrier == "load_shedding"].sum(axis=1)
            hourly_df["load_shed_MW"] = shed.values
        else:
            hourly_df["load_shed_MW"] = 0.0

        # ----------------------------------------------------
        # GENERATION BY CARRIER (EU)
        # ----------------------------------------------------
        gen_by_carrier = gen_p.T.groupby(n.generators.carrier).sum().T
        for car in gen_by_carrier.columns:
            hourly_df[f"gen_{car}_MW"] = gen_by_carrier[car].values

        # ----------------------------------------------------
        # EI03 EXTENSIONS — H2 + EXPORTS
        # ----------------------------------------------------
        elec_export_MW = (
            n.links_t.p0.loc[idx, n.links.carrier == "DC"].sum(axis=1)
            if (n.links.carrier == "DC").any()
            else pd.Series(0.0, index=idx)
        )

        elec_to_h2_MW = (
            n.links_t.p0.loc[idx, n.links.carrier == "electrolysis"].sum(axis=1)
            if (n.links.carrier == "electrolysis").any()
            else pd.Series(0.0, index=idx)
        )

        h2_produced_MW = (
            -n.links_t.p1.loc[idx, n.links.carrier == "electrolysis"].sum(axis=1)
            if (n.links.carrier == "electrolysis").any()
            else pd.Series(0.0, index=idx)
        )

        hourly_df["elec_export_MW"] = elec_export_MW.values
        hourly_df["elec_to_h2_MW"]  = elec_to_h2_MW.values
        hourly_df["h2_produced_MW"] = h2_produced_MW.values

        rows_hourly.append(hourly_df)

        # ----------------------------------------------------
        # DAILY METRICS
        # ----------------------------------------------------
        rows_daily.append({
            "date": d.date(),
            "demand_MWh": total_demand_MW.sum(),
            "generation_MWh": total_generation_MW.sum(),
            "avg_price_all_EUR_per_MWh": prices.stack().mean(),
            "load_shed_MWh": hourly_df["load_shed_MW"].sum(),
            "elec_export_MWh": elec_export_MW.sum(),
            "elec_to_h2_MWh": elec_to_h2_MW.sum(),
            "h2_produced_MWh": h2_produced_MW.sum(),
        })

    # --------------------------------------------------------
    # SAVE MONTH
    # --------------------------------------------------------
    out_dir = out_root / month
    out_dir.mkdir(parents=True, exist_ok=True)

    pd.DataFrame(rows_daily).set_index("date").to_csv(
        out_dir / f"daily_metrics_{month}.csv"
    )

    if rows_hourly:
        pd.concat(rows_hourly, ignore_index=True).to_csv(
            out_dir / f"hourly_power_{month}.csv",
            index=False
        )

    print(f"✓ Saved {month}")

print("\n=== FINAL M3_EI03 COMPLETED SUCCESSFULLY ===")


=== FINAL M3_EI03 — COUNTRY-CORRECT (MULTI-VECTOR) ===
Network: C:\Users\franc\pypsa\pypsa-eur\results\networks\EI03_energy_island.nc

=== Month 2013-01 ===


INFO:pypsa.io:Imported network EI03_energy_island.nc has buses, carriers, generators, lines, links, loads, storage_units, stores


→ 2013-01-01 (24 h)


INFO:linopy.model: Solve problem using Cbc solver
INFO:linopy.model:Solver options:
 - threads: 4
INFO:linopy.io: Writing time: 0.63s
INFO:linopy.solvers:Welcome to the CBC MILP Solver 
Version: 2.10.12 
Build Date: Aug  2 2025 

command line - cbc -printingOptions all -import C:\Users\franc\AppData\Local\Temp\linopy-problem-k57rbfer.lp -threads 4 -solve -solu C:\Users\franc\AppData\Local\Temp\linopy-solve-dpnbwtw6.sol (default strategy 1)
Option for printingOptions changed from normal to all
No match for threads - ? for list of commands
No match for 4 - ? for list of commands
Presolve 7258 (-62778) rows, 16737 (-17083) columns and 33903 (-85073) elements
Perturbing problem by 0.001% of 763.73472 - largest nonzero change 0.00086369541 ( 0.11070161%) - largest zero change 0.00086368374
0  Obj 80305408 Primal inf 16262335 (2061) Dual inf 7456.1598 (1)
220  Obj 45655411 Primal inf 18372332 (2035)
440  Obj 46692108 Primal inf 18882726 (2113)
660  Obj 46767961 Primal inf 23001459 (2182)
880

→ 2013-01-02 (24 h)


...

→ 2013-12-30 (24 h)


INFO:linopy.model: Solve problem using Cbc solver
INFO:linopy.model:Solver options:
 - threads: 4
INFO:linopy.io: Writing time: 0.67s
INFO:linopy.solvers:Welcome to the CBC MILP Solver 
Version: 2.10.12 
Build Date: Aug  2 2025 

command line - cbc -printingOptions all -import C:\Users\franc\AppData\Local\Temp\linopy-problem-4970dq27.lp -threads 4 -solve -solu C:\Users\franc\AppData\Local\Temp\linopy-solve-gew01xx9.sol (default strategy 1)
Option for printingOptions changed from normal to all
No match for threads - ? for list of commands
No match for 4 - ? for list of commands
Presolve 7277 (-62759) rows, 16742 (-17078) columns and 33925 (-85051) elements
Perturbing problem by 0.001% of 763.73472 - largest nonzero change 0.00086369541 ( 0.10765247%) - largest zero change 0.00086368374
0  Obj 88439520 Primal inf 16913530 (2088) Dual inf 26942.992 (3)
220  Obj 46439905 Primal inf 20162368 (2050)
440  Obj 47784939 Primal inf 46053450 (2161)
660  Obj 47985545 Primal inf 17421881 (2278)
880

→ 2013-12-31 (24 h)


INFO:linopy.model: Solve problem using Cbc solver
INFO:linopy.model:Solver options:
 - threads: 4
INFO:linopy.io: Writing time: 0.66s
INFO:linopy.solvers:Welcome to the CBC MILP Solver 
Version: 2.10.12 
Build Date: Aug  2 2025 

command line - cbc -printingOptions all -import C:\Users\franc\AppData\Local\Temp\linopy-problem-2cty8gq9.lp -threads 4 -solve -solu C:\Users\franc\AppData\Local\Temp\linopy-solve-l0xzxhjm.sol (default strategy 1)
Option for printingOptions changed from normal to all
No match for threads - ? for list of commands
No match for 4 - ? for list of commands
Presolve 7283 (-62753) rows, 16914 (-16906) columns and 34109 (-84867) elements
Perturbing problem by 0.001% of 763.73472 - largest nonzero change 0.00086367438 ( 0.091061246%) - largest zero change 0.0008635744
0  Obj 82290723 Primal inf 16798554 (2095) Dual inf 7456.1599 (1)
220  Obj 47640725 Primal inf 19345477 (2057)
440  Obj 48562879 Primal inf 19752040 (2128)
660  Obj 48574127 Primal inf 20221891 (2234)
880

✓ Saved 2013-12

=== FINAL M3_EI03 COMPLETED SUCCESSFULLY ===


In [16]:
# ============================================================
# FINAL M4_EI03 — YEAR AGGREGATION (2013)
# ENERGY ISLAND SCENARIO (MULTI-VECTOR)
# FULL DROP-IN — BASELINE ALIGNED + H2 EXTENSIONS
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------
root = Path(r"C:\Users\franc\pypsa\pypsa-eur")

out_root = root / "results" / "daily_runs_EI03"
ei03_dir = out_root / "EI03_2013"
ei03_dir.mkdir(parents=True, exist_ok=True)

print("M4_EI03 writing to:", ei03_dir.resolve())

# ------------------------------------------------------------
# LOAD DAILY + HOURLY FILES
# ------------------------------------------------------------
months = [f"2013-{m:02d}" for m in range(1, 13)]

daily_frames = []
hourly_frames = []

for mm in months:
    dfn = out_root / mm / f"daily_metrics_{mm}.csv"
    hfn = out_root / mm / f"hourly_power_{mm}.csv"

    if dfn.exists():
        daily_frames.append(
            pd.read_csv(dfn, parse_dates=["date"]).set_index("date")
        )
    else:
        print(f"[WARN] Missing daily file: {dfn}")

    if hfn.exists():
        hourly_frames.append(
            pd.read_csv(hfn, parse_dates=["snapshot", "date"])
        )
    else:
        print(f"[WARN] Missing hourly file: {hfn}")

if not daily_frames or not hourly_frames:
    raise RuntimeError("Missing daily or hourly data — run M3_EI03 first.")

daily = pd.concat(daily_frames).sort_index()
hourly = pd.concat(hourly_frames, ignore_index=True)

print(f"Loaded {len(daily)} daily rows.")
print(f"Loaded {len(hourly)} hourly rows.")

# ============================================================
# ANNUAL KPIs (EU — SAME AS BASELINE)
# ============================================================

demand_total = daily["demand_MWh"].sum()
generation_total = daily["generation_MWh"].sum()

variable_cost_total = (
    daily["variable_cost_EUR"].sum()
    if "variable_cost_EUR" in daily.columns else 0.0
)

avg_price_mean = (
    daily["avg_price_all_EUR_per_MWh"].mean()
    if "avg_price_all_EUR_per_MWh" in daily.columns else np.nan
)

total_load_shed = (
    daily["load_shed_MWh"].sum()
    if "load_shed_MWh" in daily.columns else 0.0
)

days_with_shedding = (
    int((daily["load_shed_MWh"] > 0).sum())
    if "load_shed_MWh" in daily.columns else 0
)

# ============================================================
# ENERGY ISLAND METRICS (EI03 EXTENSION)
# ============================================================

elec_export_MWh = daily.get("elec_export_MWh", 0.0).sum()
elec_to_h2_MWh  = daily.get("elec_to_h2_MWh", 0.0).sum()
h2_prod_MWh     = daily.get("h2_produced_MWh", 0.0).sum()

# ============================================================
# COUNTRIES MAIN — BASELINE-ALIGNED (+ EI03 EXTENSIONS)
# ============================================================

MAIN_COUNTRIES = ["DK", "DE", "NL", "NO", "SE", "BE", "FR", "PL", "PT"]
rows = []

# ----------------------------------------
# Identify load shedding hours (EU)
# ----------------------------------------
if "load_shed_MW" in hourly.columns:
    shed_ts = hourly["load_shed_MW"]
    shedding_hours = shed_ts[shed_ts > 0].index
else:
    shedding_hours = []

# ----------------------------------------
# Precompute demand during shedding hours
# ----------------------------------------
demand_during_shed = {}
total_demand_shed = 0.0

for c in MAIN_COUNTRIES:
    dcol = f"demand_{c}_MW"
    if dcol in hourly.columns and len(shedding_hours) > 0:
        val = hourly.loc[shedding_hours, dcol].sum()
    else:
        val = 0.0
    demand_during_shed[c] = val
    total_demand_shed += val

# ----------------------------------------
# Build table (IDENTICAL to baseline)
# ----------------------------------------
for c in MAIN_COUNTRIES + ["EU"]:

    if c == "EU":
        demand = demand_total
        generation = generation_total
        price = avg_price_mean
        load_shed = total_load_shed

        elec_to_h2 = elec_to_h2_MWh
        h2_prod = h2_prod_MWh

    else:
        dcol = f"demand_{c}_MW"
        gcol = f"generation_{c}_MW"
        pcol = f"price_{c}_EUR_per_MWh"

        demand = hourly[dcol].sum() if dcol in hourly.columns else 0.0
        generation = hourly[gcol].sum() if gcol in hourly.columns else 0.0

        # Load-weighted price
        if dcol in hourly.columns and pcol in hourly.columns:
            w = hourly[dcol].replace(0, np.nan)
            mask = w > 1e-6
            price = (hourly.loc[mask, pcol] * w.loc[mask]).sum() / w.loc[mask].sum()
        else:
            price = np.nan

        # Load shedding allocation
        if total_demand_shed > 0:
            load_shed = total_load_shed * demand_during_shed[c] / total_demand_shed
        else:
            load_shed = 0.0

        elec_to_h2 = 0.0
        h2_prod = 0.0

    rows.append({
        "country": c,
        "annual_demand_MWh": demand,
        "annual_generation_MWh": generation,
        "avg_price_EUR_per_MWh": price,
        "load_shed_MWh": load_shed,
        # EI03 extensions
        "elec_to_h2_MWh": elec_to_h2,
        "h2_produced_MWh": h2_prod,
    })

countries_main = pd.DataFrame(rows).set_index("country")

countries_main_out = ei03_dir / "EI03_2013_countries_main.csv"
countries_main.to_csv(countries_main_out)

print(" -", countries_main_out.name)
# ============================================================
# COUNTRIES MAIN — BASELINE-ALIGNED (+ EI03 EXTENSIONS)
# ============================================================

MAIN_COUNTRIES = ["DK", "DE", "NL", "NO", "SE", "BE", "FR", "PL", "PT"]
rows = []

# ----------------------------------------
# Identify load shedding hours (EU)
# ----------------------------------------
if "load_shed_MW" in hourly.columns:
    shed_ts = hourly["load_shed_MW"]
    shedding_hours = shed_ts[shed_ts > 0].index
else:
    shedding_hours = []

# ----------------------------------------
# Precompute demand during shedding hours
# ----------------------------------------
demand_during_shed = {}
total_demand_shed = 0.0

for c in MAIN_COUNTRIES:
    dcol = f"demand_{c}_MW"
    if dcol in hourly.columns and len(shedding_hours) > 0:
        val = hourly.loc[shedding_hours, dcol].sum()
    else:
        val = 0.0
    demand_during_shed[c] = val
    total_demand_shed += val

# ----------------------------------------
# Build table (IDENTICAL to baseline)
# ----------------------------------------
for c in MAIN_COUNTRIES + ["EU"]:

    if c == "EU":
        demand = demand_total
        generation = generation_total
        price = avg_price_mean
        load_shed = total_load_shed

        elec_to_h2 = elec_to_h2_MWh
        h2_prod = h2_prod_MWh

    else:
        dcol = f"demand_{c}_MW"
        gcol = f"generation_{c}_MW"
        pcol = f"price_{c}_EUR_per_MWh"

        demand = hourly[dcol].sum() if dcol in hourly.columns else 0.0
        generation = hourly[gcol].sum() if gcol in hourly.columns else 0.0

        # Load-weighted price
        if dcol in hourly.columns and pcol in hourly.columns:
            w = hourly[dcol].replace(0, np.nan)
            mask = w > 1e-6
            price = (hourly.loc[mask, pcol] * w.loc[mask]).sum() / w.loc[mask].sum()
        else:
            price = np.nan

        # Load shedding allocation
        if total_demand_shed > 0:
            load_shed = total_load_shed * demand_during_shed[c] / total_demand_shed
        else:
            load_shed = 0.0

        elec_to_h2 = 0.0
        h2_prod = 0.0

    rows.append({
        "country": c,
        "annual_demand_MWh": demand,
        "annual_generation_MWh": generation,
        "avg_price_EUR_per_MWh": price,
        "load_shed_MWh": load_shed,
        # EI03 extensions
        "elec_to_h2_MWh": elec_to_h2,
        "h2_produced_MWh": h2_prod,
    })

countries_main = pd.DataFrame(rows).set_index("country")

countries_main_out = ei03_dir / "EI03_2013_countries_main.csv"
countries_main.to_csv(countries_main_out)

print(" -", countries_main_out.name)


# ============================================================
# GENERATION MIX (EU — SAME AS BASELINE)
# ============================================================

gen_cols = [c for c in hourly.columns if c.startswith("gen_") and c.endswith("_MW")]

gen_by_carrier = (
    hourly[gen_cols]
    .sum()
    .rename(lambda c: c.replace("gen_", "").replace("_MW", ""))
    .drop([c for c in gen_cols if "load_shedding" in c], errors="ignore")
)

gen_mix_table = pd.DataFrame({
    "MWh": gen_by_carrier,
    "%_of_served_generation": 100.0 * gen_by_carrier / gen_by_carrier.sum()
})

# ============================================================
# CURTAILMENT & NET EXPORTS (SAFE PLACEHOLDERS — BASELINE-COMPATIBLE)
# ============================================================

# Same behaviour as baseline: empty series, but file must exist
curtailment_by_carrier = pd.Series(dtype=float, name="MWh")
net_exports_annual     = pd.Series(dtype=float, name="MWh")


# ============================================================
# SUMMARY TABLE
# ============================================================

annual_summary_df = pd.DataFrame([{
    "demand_MWh_total": demand_total,
    "generation_MWh_total": generation_total,
    "variable_cost_EUR_total": variable_cost_total,
    "avg_price_EUR_per_MWh_mean": avg_price_mean,
    "total_load_shed_MWh": total_load_shed,
    "days_with_any_load_shed": days_with_shedding,
    # EI03
    "elec_export_MWh_total": elec_export_MWh,
    "elec_to_h2_MWh_total": elec_to_h2_MWh,
    "h2_produced_MWh_total": h2_prod_MWh,
}])

# ------------------------------------------------------------
# SAVE FILES (SAME NAMING PATTERN)
# ------------------------------------------------------------
daily_out   = ei03_dir / "EI03_2013_daily.csv"
summary_out = ei03_dir / "EI03_2013_summary.csv"
genmix_out  = ei03_dir / "EI03_2013_generation_mix.csv"
curt_out    = ei03_dir / "EI03_2013_curtailment.csv"
netexp_out  = ei03_dir / "EI03_2013_net_exports.csv"

daily.to_csv(daily_out)
annual_summary_df.to_csv(summary_out, index=False)
gen_mix_table.to_csv(genmix_out)
curtailment_by_carrier.to_csv(curt_out)
net_exports_annual.to_csv(netexp_out)


# ============================================================
# FINAL REPORT — FILES WRITTEN (EI03)
# ============================================================

written_files = [
    daily_out,
    summary_out,
    genmix_out,
    curt_out,
    netexp_out,
    countries_main_out,
]

print("\n✔ EI03 M4 COMPLETE — FILES WRITTEN / UPDATED:")
for f in written_files:
    print(" -", f.name)

print("\n=== EI03 M4 COMPLETED SUCCESSFULLY ===")


M4_EI03 writing to: C:\Users\franc\pypsa\pypsa-eur\results\daily_runs_EI03\EI03_2013
Loaded 365 daily rows.
Loaded 8760 hourly rows.
 - EI03_2013_countries_main.csv
 - EI03_2013_countries_main.csv

✔ EI03 M4 COMPLETE — FILES WRITTEN / UPDATED:
 - EI03_2013_daily.csv
 - EI03_2013_summary.csv
 - EI03_2013_generation_mix.csv
 - EI03_2013_curtailment.csv
 - EI03_2013_net_exports.csv
 - EI03_2013_countries_main.csv

=== EI03 M4 COMPLETED SUCCESSFULLY ===


In [2]:
# ============================================================
# FINAL M5_EI03 — CALENDAR 8 CATEGORIES (2013)
# FULL DROP-IN — BASELINE ALIGNED
# CREATES: labels_by_day, category_summary, country_categories
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import date

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------
SCENARIO = "EI03"
root = Path(r"C:\Users\franc\pypsa\pypsa-eur")

out_root = root / "results" / f"daily_runs_{SCENARIO}"
assert out_root.exists(), f"OUTDIR not found: {out_root}"

clusters_dir = out_root / "clusters" / "calendar_8cats_2013"
clusters_dir.mkdir(parents=True, exist_ok=True)

ei03_dir = out_root / "EI03_2013"
ei03_dir.mkdir(parents=True, exist_ok=True)

written_files = []

print("M5_EI03 writing to:", clusters_dir.resolve())

# ------------------------------------------------------------
# LOAD HOURLY FILES (ALL MONTHS)
# ------------------------------------------------------------
frames = []

for m in range(1, 13):
    mm = f"2013-{m:02d}"
    fn = out_root / mm / f"hourly_power_{mm}.csv"
    if not fn.exists():
        raise FileNotFoundError(fn)

    df = pd.read_csv(fn, parse_dates=["snapshot", "date"])
    df["hour"] = df["snapshot"].dt.hour
    frames.append(df)

hourly = pd.concat(frames, ignore_index=True)
print(f"Loaded {len(hourly)} hourly rows")

# ------------------------------------------------------------
# SEASON FUNCTION (IDENTICAL TO BASELINE)
# ------------------------------------------------------------
def season_2013(d):
    if date(2013,1,1) <= d <= date(2013,3,19):
        return "Winter"
    if date(2013,3,20) <= d <= date(2013,6,20):
        return "Spring"
    if date(2013,6,21) <= d <= date(2013,9,21):
        return "Summer"
    if date(2013,9,22) <= d <= date(2013,12,20):
        return "Autumn"
    return "Winter"

# ------------------------------------------------------------
# LABELS BY DAY
# ------------------------------------------------------------
labels = []

for d in sorted(hourly["date"].unique()):
    d_date = pd.Timestamp(d).date()
    season = season_2013(d_date)
    wwe = "weekend" if pd.Timestamp(d).weekday() >= 5 else "weekday"

    labels.append({
        "date": d,
        "season": season,
        "weekday_or_weekend": wwe,
        "category": f"{season.lower()}_{wwe}",
    })

labels_df = pd.DataFrame(labels).set_index("date")

f = clusters_dir / "labels_by_day.csv"
labels_df.to_csv(f)
written_files.append(f.name)

hourly = hourly.merge(labels_df, left_on="date", right_index=True, how="left")

# ------------------------------------------------------------
# DETECT VRE (EU) — SAME LOGIC AS BASELINE
# ------------------------------------------------------------
vre_cols = [c for c in hourly.columns if c.startswith("gen_") and c.endswith("_MW")]
hourly["vre_total_MW"] = hourly[vre_cols].sum(axis=1)

# ------------------------------------------------------------
# COUNTRY × CATEGORY SUMMARY (CORE OUTPUT)
# ------------------------------------------------------------
MAIN_COUNTRIES = ["DK", "DE", "NL", "NO", "SE", "BE", "FR", "PL", "PT"]
rows = []

for cat, days in labels_df.groupby("category").groups.items():
    hsub = hourly[hourly["date"].isin(days)]

    total_demand_eu = hsub["total_demand_MW"].mean()

    for c in MAIN_COUNTRIES + ["EU"]:

        if c == "EU":
            demand = total_demand_eu
            vre = hsub["vre_total_MW"].mean()

            # demand-weighted EU price (IDENTICAL to baseline)
            num = 0.0
            den = 0.0
            for cc in MAIN_COUNTRIES:
                dcol = f"demand_{cc}_MW"
                pcol = f"price_{cc}_EUR_per_MWh"
                if dcol in hsub.columns and pcol in hsub.columns:
                    w = hsub[dcol].mean()
                    num += w * hsub[pcol].mean()
                    den += w
            price = num / den if den > 0 else np.nan

        else:
            dcol = f"demand_{c}_MW"
            pcol = f"price_{c}_EUR_per_MWh"

            demand = hsub[dcol].mean() if dcol in hsub.columns else 0.0
            share = demand / total_demand_eu if total_demand_eu > 0 else 0.0
            vre = hsub["vre_total_MW"].mean() * share

            price = hsub[pcol].mean() if pcol in hsub.columns else np.nan

        rows.append({
            "country": c,
            "category": cat,
            "season": labels_df.loc[days[0], "season"],
            "weekday_or_weekend": labels_df.loc[days[0], "weekday_or_weekend"],
            "n_days": len(days),
            "mean_daily_demand_MW": demand,
            "mean_daily_vre_MW": vre,
            "mean_price_EUR_per_MWh": price,
        })

country_categories = pd.DataFrame(rows)

f = ei03_dir / "EI03_2013_country_categories.csv"
country_categories.to_csv(f, index=False)
written_files.append(f.name)

# ------------------------------------------------------------
# CATEGORY SUMMARY (EU-LEVEL)
# ------------------------------------------------------------
summary_rows = []

for cat, days in labels_df.groupby("category").groups.items():
    hsub = hourly[hourly["date"].isin(days)]

    summary_rows.append({
        "category": cat,
        "season": labels_df.loc[days[0], "season"],
        "weekday_or_weekend": labels_df.loc[days[0], "weekday_or_weekend"],
        "n_days": len(days),
        "mean_price_EUR_per_MWh": hsub["total_demand_MW"].mean(),
    })

f = clusters_dir / "category_summary.csv"
pd.DataFrame(summary_rows).set_index("category").to_csv(f)
written_files.append(f.name)

# ------------------------------------------------------------
# FINAL REPORT
# ------------------------------------------------------------
print("\n✔ EI03 M5 COMPLETE — FILES WRITTEN:")
for fn in written_files:
    print(" -", fn)

print("\n=== EI03 M5 COMPLETED SUCCESSFULLY ===")


M5_EI03 writing to: C:\Users\franc\pypsa\pypsa-eur\results\daily_runs_EI03\clusters\calendar_8cats_2013
Loaded 8760 hourly rows

✔ EI03 M5 COMPLETE — FILES WRITTEN:
 - labels_by_day.csv
 - EI03_2013_country_categories.csv
 - category_summary.csv

=== EI03 M5 COMPLETED SUCCESSFULLY ===


In [3]:
# ============================================================
# FINAL M6_EI03 — REBUILD VRE × CATEGORY (+ H2 EU) (2013)
# ENERGY ISLAND SCENARIO
#
# SOURCE OF TRUTH:
#   - Hourly dispatch → gen_*_MW, elec_to_h2_MW, h2_produced_MW (from M3_EI03)
#   - Category labels → labels_by_day.csv (from M5)
#
# OUTPUT:
#   - EI03_2013_vre_by_category.csv
#   - EI03_2013_h2_by_category.csv   (EU only)
# ============================================================

import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------
root = Path(r"C:\Users\franc\pypsa\pypsa-eur")
out_root = root / "results" / "daily_runs_EI03"

clusters_dir = out_root / "clusters" / "calendar_8cats_2013"
ei03_dir = out_root / "EI03_2013"
ei03_dir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# LOAD CATEGORY LABELS
# ------------------------------------------------------------
labels = (
    pd.read_csv(
        clusters_dir / "labels_by_day.csv",
        parse_dates=["date"]
    )
    .set_index("date")
)

# ------------------------------------------------------------
# LOAD ALL HOURLY FILES
# ------------------------------------------------------------
hourly_frames = []

for m in range(1, 13):
    mm = f"2013-{m:02d}"
    fn = out_root / mm / f"hourly_power_{mm}.csv"

    if not fn.exists():
        print(f"[WARN] Missing hourly file: {fn}")
        continue

    df = pd.read_csv(fn, parse_dates=["snapshot"])
    df["date"] = df["snapshot"].dt.normalize()
    hourly_frames.append(df)

if not hourly_frames:
    raise RuntimeError("No hourly files found for EI03 scenario.")

hourly = pd.concat(hourly_frames, ignore_index=True)

print(f"Loaded {len(hourly)} hourly rows.")

# ------------------------------------------------------------
# DETECT VRE GENERATION COLUMNS (SAME AS BASELINE)
# ------------------------------------------------------------
vre_carriers = [
    "onwind",
    "offwind-ac",
    "offwind-dc",
    "offwind-float",
    "solar",
    "solar-hsat",
    "ror",
]

vre_cols = [
    c for c in hourly.columns
    if c.startswith("gen_")
    and c.endswith("_MW")
    and any(c == f"gen_{vc}_MW" for vc in vre_carriers)
]

if not vre_cols:
    raise RuntimeError("No VRE generation columns found in hourly data.")

print("Detected VRE technologies:", vre_cols)

# ------------------------------------------------------------
# MERGE CATEGORY LABELS
# ------------------------------------------------------------
hourly = hourly.merge(
    labels[["category", "season", "weekday_or_weekend"]],
    left_on="date",
    right_index=True,
    how="inner"
)

# ============================================================
# VRE × TECHNOLOGY × CATEGORY (IDENTICAL TO BASELINE)
# ============================================================
vre_rows = []

for col in vre_cols:
    tech = col.replace("gen_", "").replace("_MW", "")

    for cat, g in hourly.groupby("category"):
        vre_rows.append({
            "technology": tech,
            "category": cat,
            "season": g["season"].iloc[0],
            "weekday_or_weekend": g["weekday_or_weekend"].iloc[0],
            "n_hours": len(g),
            "mean_vre_MW": g[col].mean(),
            "mean_daily_vre_MWh": g[col].mean() * 24.0,
        })

vre_by_category = (
    pd.DataFrame(vre_rows)
    .set_index(["technology", "category"])
    .sort_index()
)

vre_out = ei03_dir / "EI03_2013_vre_by_category.csv"
vre_by_category.to_csv(vre_out)

# ============================================================
# H2 × CATEGORY (EU ONLY — EI03 EXTENSION)
# ============================================================
required_h2_cols = ["elec_to_h2_MW", "h2_produced_MW"]

missing = [c for c in required_h2_cols if c not in hourly.columns]
if missing:
    raise RuntimeError(f"Missing H2 columns in hourly data: {missing}")

h2_rows = []

for cat, g in hourly.groupby("category"):
    h2_rows.append({
        "category": cat,
        "season": g["season"].iloc[0],
        "weekday_or_weekend": g["weekday_or_weekend"].iloc[0],
        "n_hours": len(g),
        "mean_elec_to_h2_MW": g["elec_to_h2_MW"].mean(),
        "mean_h2_produced_MW": g["h2_produced_MW"].mean(),
        "mean_daily_elec_to_h2_MWh": g["elec_to_h2_MW"].mean() * 24.0,
        "mean_daily_h2_produced_MWh": g["h2_produced_MW"].mean() * 24.0,
    })

h2_by_category = (
    pd.DataFrame(h2_rows)
    .set_index("category")
    .sort_index()
)

h2_out = ei03_dir / "EI03_2013_h2_by_category.csv"
h2_by_category.to_csv(h2_out)

# ------------------------------------------------------------
# FINAL REPORT
# ------------------------------------------------------------
print("\n✔ EI03 M6 COMPLETE — FILES WRITTEN:")
print(" -", vre_out.name)
print(" -", h2_out.name)

print("\n=== EI03 M6 COMPLETED SUCCESSFULLY ===")


Loaded 8760 hourly rows.
Detected VRE technologies: ['gen_offwind-ac_MW', 'gen_offwind-dc_MW', 'gen_offwind-float_MW', 'gen_onwind_MW', 'gen_ror_MW', 'gen_solar_MW', 'gen_solar-hsat_MW']

✔ EI03 M6 COMPLETE — FILES WRITTEN:
 - EI03_2013_vre_by_category.csv
 - EI03_2013_h2_by_category.csv

=== EI03 M6 COMPLETED SUCCESSFULLY ===


In [2]:
# ============================================================
# FINAL M7 — ECONOMIC ACCOUNTING BY CATEGORY (2013)
# GENERIC: EI01 (Export Hub) + EI03 (Multi-vector)
#
# SOURCES:
#   - Hourly dispatch (M3)
#   - Calendar categories (M5)
#
# OUTPUTS:
#   - *_2013_costs_by_category.csv
#   - *_2013_annual_costs.csv
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
SCENARIO =  "EI03"   # or "EI01"

root = Path(r"C:\Users\franc\pypsa\pypsa-eur")
out_root = root / "results" / f"daily_runs_{SCENARIO}"

clusters_dir = out_root / "clusters" / "calendar_8cats_2013"
year_dir = out_root / f"{SCENARIO}_2013"
year_dir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# ECONOMIC PARAMETERS
# ------------------------------------------------------------
COSTS = {
    "offwind": 0.0,
    "hvdc": 1.0,
    "electrolysis": 60.0,      # €/MWh_el
    "h2_storage": 0.1,         # €/MWh_H2
    "curtailment": 5.0,        # €/MWh
    "load_shedding": 10_000.0,
}

H2_PRICE = 90.0   # €/MWh_H2

# ------------------------------------------------------------
# LOAD CATEGORY LABELS
# ------------------------------------------------------------
labels = (
    pd.read_csv(
        clusters_dir / "labels_by_day.csv",
        parse_dates=["date"]
    )
    .set_index("date")
)

# ------------------------------------------------------------
# LOAD ALL HOURLY FILES
# ------------------------------------------------------------
frames = []

for m in range(1, 13):
    mm = f"2013-{m:02d}"
    fn = out_root / mm / f"hourly_power_{mm}.csv"
    if not fn.exists():
        raise FileNotFoundError(fn)

    df = pd.read_csv(fn, parse_dates=["snapshot"])
    df["date"] = df["snapshot"].dt.normalize()
    frames.append(df)

hourly = pd.concat(frames, ignore_index=True)
print(f"Loaded {len(hourly)} hourly rows")

# ------------------------------------------------------------
# MERGE CATEGORIES
# ------------------------------------------------------------
hourly = hourly.merge(
    labels[["category", "season", "weekday_or_weekend"]],
    left_on="date",
    right_index=True,
    how="inner"
)

# ------------------------------------------------------------
# DETECT SCENARIO FEATURES
# ------------------------------------------------------------
has_h2 = {"elec_to_h2_MW", "h2_produced_MW"}.issubset(hourly.columns)
has_export = "elec_export_MW" in hourly.columns

print("H2 active:", has_h2)
print("Electric export active:", has_export)

# ------------------------------------------------------------
# DETECT OFFSHORE WIND
# ------------------------------------------------------------
offwind_cols = [
    c for c in hourly.columns
    if c.startswith("gen_") and "offwind" in c
]

if not offwind_cols:
    raise RuntimeError("No offshore wind generation detected")

# ------------------------------------------------------------
# DETECT PRICES BY COUNTRY
# ------------------------------------------------------------
price_cols = {
    c.replace("price_", "").replace("_EUR_per_MWh", ""): c
    for c in hourly.columns
    if c.startswith("price_") and c.endswith("_EUR_per_MWh")
}

# ------------------------------------------------------------
# MAIN AGGREGATION
# ------------------------------------------------------------
rows = []

for cat, g in hourly.groupby("category"):

    out = {
        "category": cat,
        "season": g["season"].iloc[0],
        "weekday_or_weekend": g["weekday_or_weekend"].iloc[0],
        "n_hours": len(g),
    }

    # -----------------------------
    # Offshore wind
    # -----------------------------
    offwind_MWh = g[offwind_cols].sum(axis=1).sum()
    out["offwind_MWh"] = offwind_MWh
    out["cost_offwind_EUR"] = offwind_MWh * COSTS["offwind"]

    # -----------------------------
    # Curtailment (proxy)
    # -----------------------------
    if "p_max_pu" in g.columns:
        # optional if ever exported
        curtailment_MWh = g["p_max_pu"].sum() - offwind_MWh
    else:
        curtailment_MWh = 0.0

    out["curtailment_MWh"] = max(curtailment_MWh, 0.0)
    out["cost_curtailment_EUR"] = out["curtailment_MWh"] * COSTS["curtailment"]

    # -----------------------------
    # Load shedding
    # -----------------------------
    if "load_shed_MW" in g.columns:
        shed_MWh = g["load_shed_MW"].sum()
    else:
        shed_MWh = 0.0

    out["load_shed_MWh"] = shed_MWh
    out["cost_load_shedding_EUR"] = shed_MWh * COSTS["load_shedding"]

    # -----------------------------
    # Electric export revenue
    # -----------------------------
    revenue_elec = 0.0

    if has_export:
        for country, pcol in price_cols.items():
            # proportional allocation by price country
            revenue_elec += (
                g["elec_export_MW"] * g[pcol]
            ).sum()

    out["revenue_electricity_EUR"] = revenue_elec

    # -----------------------------
    # H2 economics (EU only)
    # -----------------------------
    if has_h2:
        h2_MWh = g["h2_produced_MW"].sum()
        elec_to_h2_MWh = g["elec_to_h2_MW"].sum()

        out["h2_produced_MWh"] = h2_MWh
        out["cost_electrolysis_EUR"] = elec_to_h2_MWh * COSTS["electrolysis"]
        out["revenue_h2_EUR"] = h2_MWh * H2_PRICE
    else:
        out["h2_produced_MWh"] = 0.0
        out["cost_electrolysis_EUR"] = 0.0
        out["revenue_h2_EUR"] = 0.0

    # -----------------------------
    # Net system value
    # -----------------------------
    out["net_system_value_EUR"] = (
        out["revenue_electricity_EUR"]
        + out["revenue_h2_EUR"]
        - out["cost_offwind_EUR"]
        - out["cost_electrolysis_EUR"]
        - out["cost_curtailment_EUR"]
        - out["cost_load_shedding_EUR"]
    )

    rows.append(out)

# ------------------------------------------------------------
# SAVE OUTPUTS
# ------------------------------------------------------------
df_cat = pd.DataFrame(rows).set_index("category").sort_index()

out_cat = year_dir / f"{SCENARIO}_2013_costs_by_category.csv"
df_cat.to_csv(out_cat)

df_annual = df_cat.sum(numeric_only=True).to_frame(name="value")
out_annual = year_dir / f"{SCENARIO}_2013_annual_costs.csv"
df_annual.to_csv(out_annual)

# ------------------------------------------------------------
# FINAL REPORT
# ------------------------------------------------------------
print("\n✔ M7 COMPLETE — FILES WRITTEN:")
print(" -", out_cat.name)
print(" -", out_annual.name)

print("\n=== M7 COMPLETED SUCCESSFULLY ===")


Loaded 8760 hourly rows
H2 active: True
Electric export active: True

✔ M7 COMPLETE — FILES WRITTEN:
 - EI03_2013_costs_by_category.csv
 - EI03_2013_annual_costs.csv

=== M7 COMPLETED SUCCESSFULLY ===


In [3]:
# ============================================================
# FINAL M8_EI03 — CAPEX + SYSTEM VALUE (2013)
# ENERGY ISLAND — MULTI-VECTOR (ELECTRICITY + H2)
# ============================================================

import pypsa
import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
SCENARIO = "EI03"
YEAR = 2013

root = Path(r"C:\Users\franc\pypsa\pypsa-eur")

network_fn = root / "results" / "networks" / "EI03_energy_island.nc"
out_root   = root / "results" / f"daily_runs_{SCENARIO}"
scenario_dir = out_root / f"{SCENARIO}_{YEAR}"
scenario_dir.mkdir(parents=True, exist_ok=True)

print("\nM8 — loading network:")
print(" ", network_fn.name)

assert network_fn.exists(), f"Network not found: {network_fn}"

n = pypsa.Network(network_fn)

# ------------------------------------------------------------
# LOAD ANNUAL ECONOMIC RESULTS FROM M7
# ------------------------------------------------------------
annual_costs_fn = scenario_dir / f"{SCENARIO}_{YEAR}_annual_costs.csv"
assert annual_costs_fn.exists(), "Run M7_EI03 before M8."

annual_costs = pd.read_csv(annual_costs_fn)

# Robust handling: metric may be index or column
if "metric" in annual_costs.columns:
    annual_costs = annual_costs.set_index("metric")
else:
    annual_costs = annual_costs.set_index(annual_costs.columns[0])

net_system_value_before_capex = float(
    annual_costs.loc["net_system_value_EUR", "value"]
)


# ------------------------------------------------------------
# CAPEX ASSUMPTIONS (€/unit)
# ------------------------------------------------------------
CAPEX = {
    # Electricity
    "offwind_EUR_per_MW":        2_400_000,
    "hvdc_EUR_per_MW":             600_000,
    "battery_EUR_per_MWh":         400_000,

    # Hydrogen
    "electrolyser_EUR_per_MW":     800_000,
    "h2_storage_EUR_per_MWh":       40_000,
}

LIFETIME_YEARS = {
    "offwind":      25,
    "hvdc":         40,
    "battery":      15,
    "electrolyser": 20,
    "h2_storage":   30,
}

# ------------------------------------------------------------
# INSTALLED CAPACITIES
# ------------------------------------------------------------
offwind_MW = n.generators.loc[
    n.generators.carrier.str.contains("offwind"),
    "p_nom"
].sum()

hvdc_MW = n.links.loc[
    n.links.carrier == "DC",
    "p_nom"
].sum()

battery_MWh = n.storage_units.loc[
    n.storage_units.carrier == "battery",
    "p_nom"
].sum() * n.storage_units.loc[
    n.storage_units.carrier == "battery",
    "max_hours"
].mean()

electrolyser_MW = n.links.loc[
    n.links.carrier == "electrolysis",
    "p_nom"
].sum()

h2_storage_MWh = n.stores.loc[
    n.stores.carrier == "hydrogen",
    "e_nom"
].sum()

print("\nInstalled capacities:")
print(f" - Offshore wind: {offwind_MW:.1f} MW")
print(f" - HVDC:          {hvdc_MW:.1f} MW")
print(f" - Battery:       {battery_MWh:.1f} MWh")
print(f" - Electrolyser:  {electrolyser_MW:.1f} MW")
print(f" - H2 storage:    {h2_storage_MWh:.1f} MWh")

# ------------------------------------------------------------
# ANNUALISED CAPEX
# ------------------------------------------------------------
capex_offwind_EUR = (
    offwind_MW * CAPEX["offwind_EUR_per_MW"]
    / LIFETIME_YEARS["offwind"]
)

capex_hvdc_EUR = (
    hvdc_MW * CAPEX["hvdc_EUR_per_MW"]
    / LIFETIME_YEARS["hvdc"]
)

capex_battery_EUR = (
    battery_MWh * CAPEX["battery_EUR_per_MWh"]
    / LIFETIME_YEARS["battery"]
)

capex_electrolyser_EUR = (
    electrolyser_MW * CAPEX["electrolyser_EUR_per_MW"]
    / LIFETIME_YEARS["electrolyser"]
)

capex_h2_storage_EUR = (
    h2_storage_MWh * CAPEX["h2_storage_EUR_per_MWh"]
    / LIFETIME_YEARS["h2_storage"]
)

total_capex_EUR = (
    capex_offwind_EUR
    + capex_hvdc_EUR
    + capex_battery_EUR
    + capex_electrolyser_EUR
    + capex_h2_storage_EUR
)

# ------------------------------------------------------------
# FINAL SYSTEM VALUE
# ------------------------------------------------------------
net_system_value_after_capex_EUR = (
    net_system_value_before_capex - total_capex_EUR
)

# ------------------------------------------------------------
# OUTPUT TABLE
# ------------------------------------------------------------
rows = [
    ("capex_offwind_EUR", capex_offwind_EUR),
    ("capex_hvdc_EUR", capex_hvdc_EUR),
    ("capex_battery_EUR", capex_battery_EUR),
    ("capex_electrolyser_EUR", capex_electrolyser_EUR),
    ("capex_h2_storage_EUR", capex_h2_storage_EUR),
    ("total_capex_EUR", total_capex_EUR),
    ("net_system_value_before_capex_EUR", net_system_value_before_capex),
    ("net_system_value_after_capex_EUR", net_system_value_after_capex_EUR),
]

df_out = pd.DataFrame(rows, columns=["metric", "value"]).set_index("metric")

out_fn = scenario_dir / f"{SCENARIO}_{YEAR}_capex_and_system_value.csv"
df_out.to_csv(out_fn)

# ------------------------------------------------------------
# FINAL REPORT
# ------------------------------------------------------------
print("\n✔ M8_EI03 COMPLETE — CAPEX ACCOUNTING ADDED")
print(" -", out_fn.name)

print("\nSummary:")
for k, v in df_out["value"].items():
    print(f" {k:35s}: {v:,.0f} €")



M8 — loading network:
  EI03_energy_island.nc


INFO:pypsa.io:Imported network EI03_energy_island.nc has buses, carriers, generators, lines, links, loads, storage_units, stores



Installed capacities:
 - Offshore wind: 62820.6 MW
 - HVDC:          39270.0 MW
 - Battery:       2000.0 MWh
 - Electrolyser:  5000.0 MW
 - H2 storage:    300000.0 MWh

✔ M8_EI03 COMPLETE — CAPEX ACCOUNTING ADDED
 - EI03_2013_capex_and_system_value.csv

Summary:
 capex_offwind_EUR                  : 6,030,777,600 €
 capex_hvdc_EUR                     : 589,050,000 €
 capex_battery_EUR                  : 53,333,333 €
 capex_electrolyser_EUR             : 200,000,000 €
 capex_h2_storage_EUR               : 400,000,000 €
 total_capex_EUR                    : 7,273,160,933 €
 net_system_value_before_capex_EUR  : -13,601,618,493 €
 net_system_value_after_capex_EUR   : -20,874,779,427 €


In [9]:
# ============================================================
# FINAL M9 — WELFARE & PRICE IMPACT VS BASELINE (2013)
# COMPARES: Baseline vs Energy Island (EI01 or EI03)
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
SCENARIO = "EI03"   # "EI01" or "EI03"
YEAR = 2013

MAIN_COUNTRIES = ["DK", "DE", "NL", "NO", "SE", "BE", "FR", "PL", "PT"]

VOLL = 10_000.0  # €/MWh (Value of Lost Load)

# ------------------------------------------------------------
# PATHS (FIXED & ROBUST)
# ------------------------------------------------------------
root = Path(r"C:\Users\franc\pypsa\pypsa-eur")

baseline_dir = root / "results" / "daily_runs" / "baseline_nohub_2013"
scenario_dir = root / "results" / f"daily_runs_{SCENARIO}" / f"{SCENARIO}_2013"

assert baseline_dir.exists(), baseline_dir
assert scenario_dir.exists(), scenario_dir

# ------------------------------------------------------------
# LOAD DATA
# ------------------------------------------------------------
baseline = pd.read_csv(
    baseline_dir / "baseline_nohub_2013_countries_main.csv",
    index_col="country"
)

scenario = pd.read_csv(
    scenario_dir / f"{SCENARIO}_2013_countries_main.csv",
    index_col="country"
)

# ------------------------------------------------------------
# SANITY CHECKS
# ------------------------------------------------------------
assert all(c in baseline.index for c in MAIN_COUNTRIES)
assert all(c in scenario.index for c in MAIN_COUNTRIES)

# ------------------------------------------------------------
# WELFARE CALCULATION
# ------------------------------------------------------------
rows = []

for c in MAIN_COUNTRIES:

    demand = baseline.loc[c, "annual_demand_MWh"]

    # -------------------------
    # PRICE EFFECT
    # -------------------------
    price_baseline = baseline.loc[c, "avg_price_EUR_per_MWh"]
    price_scenario = scenario.loc[c, "avg_price_EUR_per_MWh"]

    delta_price = price_scenario - price_baseline
    welfare_price = - delta_price * demand   # price drop = positive welfare

    # -------------------------
    # LOAD SHEDDING EFFECT
    # -------------------------
    shed_baseline = baseline.loc[c, "load_shed_MWh"]
    shed_scenario = scenario.loc[c, "load_shed_MWh"]

    delta_shed = shed_scenario - shed_baseline
    welfare_shed = - delta_shed * VOLL        # less shedding = positive welfare

    # -------------------------
    # TOTAL
    # -------------------------
    total_welfare = welfare_price + welfare_shed

    rows.append({
        "country": c,
        "delta_price_EUR_per_MWh": delta_price,
        "welfare_price_EUR": welfare_price,
        "delta_load_shed_MWh": delta_shed,
        "welfare_load_shed_EUR": welfare_shed,
        "total_welfare_EUR": total_welfare,
    })

welfare_df = pd.DataFrame(rows).set_index("country")

# ------------------------------------------------------------
# EU AGGREGATE
# ------------------------------------------------------------
welfare_df.loc["EU"] = {
    "delta_price_EUR_per_MWh": np.nan,
    "welfare_price_EUR": welfare_df["welfare_price_EUR"].sum(),
    "delta_load_shed_MWh": welfare_df["delta_load_shed_MWh"].sum(),
    "welfare_load_shed_EUR": welfare_df["welfare_load_shed_EUR"].sum(),
    "total_welfare_EUR": welfare_df["total_welfare_EUR"].sum(),
}

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------
out_fn = scenario_dir / f"{SCENARIO}_2013_welfare_vs_baseline.csv"
welfare_df.to_csv(out_fn)

print("\n✔ M9 COMPLETE — WELFARE VS BASELINE")
print("Scenario:", SCENARIO)
print("Output:", out_fn)
print("\nEU total welfare gain:")
print(f"{welfare_df.loc['EU', 'total_welfare_EUR']:,.0f} €")



✔ M9 COMPLETE — WELFARE VS BASELINE
Scenario: EI03
Output: C:\Users\franc\pypsa\pypsa-eur\results\daily_runs_EI03\EI03_2013\EI03_2013_welfare_vs_baseline.csv

EU total welfare gain:
-37,460,555,365 €


In [13]:
# ============================================================
# FINAL M10 — EU CONSUMER WELFARE (ΔCS − OPEX)
# BASELINE vs ENERGY ISLAND (EI01 / EI03)
#
# DEFINIÇÃO:
#   Year 0  : -CAPEX
#   Year 1+ : Δ Consumer Surplus EU − OPEX
#
# OUTPUT:
#   results/daily_runs_<SCENARIO>/welfares/
#     <SCENARIO>_eu_consumer_welfare.csv
# ============================================================

import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
SCENARIO = "EI03"          # "EI01" ou "EI03"
HORIZON_YEARS = 40

MAIN_COUNTRIES = ["DK","DE","NL","NO","SE","BE","FR","PL","PT"]

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------
root = Path(r"C:\Users\franc\pypsa\pypsa-eur")

baseline_dir = root / "results" / "daily_runs" / "baseline_nohub_2013"
scenario_dir = root / "results" / f"daily_runs_{SCENARIO}" / f"{SCENARIO}_2013"

welfare_dir = root / "results" / f"daily_runs_{SCENARIO}" / "welfares"
welfare_dir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# LOAD INPUT FILES
# ------------------------------------------------------------
baseline = pd.read_csv(
    baseline_dir / "baseline_nohub_2013_countries_main.csv",
    index_col="country"
)

scenario = pd.read_csv(
    scenario_dir / f"{SCENARIO}_2013_countries_main.csv",
    index_col="country"
)

annual_costs = pd.read_csv(
    scenario_dir / f"{SCENARIO}_2013_annual_costs.csv",
    index_col=0
)

capex_df = pd.read_csv(
    scenario_dir / f"{SCENARIO}_2013_capex_and_system_value.csv",
    index_col=0
)

# ============================================================
# 1) Δ CONSUMER SURPLUS (EU)
# ============================================================
delta_cs_eu = 0.0

for c in MAIN_COUNTRIES:
    demand = scenario.loc[c, "annual_demand_MWh"]
    p_base = baseline.loc[c, "avg_price_EUR_per_MWh"]
    p_scen = scenario.loc[c, "avg_price_EUR_per_MWh"]

    delta_cs_eu += (p_base - p_scen) * demand

# ============================================================
# 2) OPEX (SEM LOAD SHEDDING)
# ============================================================
opex = (
    annual_costs
    .loc[
        annual_costs.index.str.startswith("cost_")
        & (~annual_costs.index.str.contains("load_shedding")),
        "value"
    ]
    .sum()
)

# ============================================================
# 3) ANNUAL WELFARE (STEADY STATE)
# ============================================================
annual_welfare = delta_cs_eu - opex

# ============================================================
# 4) CAPEX (YEAR 0)
# ============================================================
total_capex = capex_df.loc["total_capex_EUR", "value"]

# ============================================================
# 5) INTERTEMPORAL TABLE
# ============================================================
rows = []

cumulative = -total_capex

rows.append({
    "year": 0,
    "annual_welfare_EUR": -total_capex,
    "cumulative_welfare_EUR": cumulative,
    "note": "CAPEX"
})

for y in range(1, HORIZON_YEARS + 1):
    cumulative += annual_welfare
    rows.append({
        "year": y,
        "annual_welfare_EUR": annual_welfare,
        "cumulative_welfare_EUR": cumulative,
        "note": ""
    })

df = pd.DataFrame(rows)

# ============================================================
# 6) PAYBACK YEAR
# ============================================================
payback = df[df["cumulative_welfare_EUR"] >= 0]
payback_year = int(payback.iloc[0]["year"]) if not payback.empty else None

# ============================================================
# SAVE OUTPUT
# ============================================================
out_fn = welfare_dir / f"{SCENARIO}_eu_consumer_welfare.csv"
df.to_csv(out_fn, index=False)

# ============================================================
# PRINT RESULTS
# ============================================================
print("\n=== M10 — EU CONSUMER WELFARE (ΔCS − OPEX) ===\n")
print(f"Scenario: {SCENARIO}")
print(f"Δ Consumer Surplus EU : {delta_cs_eu:,.0f} € / year")
print(f"OPEX (annual)        : {opex:,.0f} € / year")
print(f"Annual welfare       : {annual_welfare:,.0f} € / year")
print(f"CAPEX (year 0)       : {-total_capex:,.0f} €\n")

print("YEAR | Annual welfare (€) | Cumulative welfare (€)")
print("-" * 60)

for _, r in df.iterrows():
    print(
        f"{int(r.year):4d} | "
        f"{r.annual_welfare_EUR:>20,.0f} | "
        f"{r.cumulative_welfare_EUR:>26,.0f}"
    )

print("\nPayback year:", payback_year if payback_year is not None else "No payback in horizon")
print("\n✔ M10 COMPLETE — FILE WRITTEN:")
print(" -", out_fn)



=== M10 — EU CONSUMER WELFARE (ΔCS − OPEX) ===

Scenario: EI03
Δ Consumer Surplus EU : 6,252,746,449 € / year
OPEX (annual)        : 645,120,546 € / year
Annual welfare       : 5,607,625,903 € / year
CAPEX (year 0)       : -7,273,160,933 €

YEAR | Annual welfare (€) | Cumulative welfare (€)
------------------------------------------------------------
   0 |       -7,273,160,933 |             -7,273,160,933
   1 |        5,607,625,903 |             -1,665,535,031
   2 |        5,607,625,903 |              3,942,090,872
   3 |        5,607,625,903 |              9,549,716,774
   4 |        5,607,625,903 |             15,157,342,677
   5 |        5,607,625,903 |             20,764,968,579
   6 |        5,607,625,903 |             26,372,594,482
   7 |        5,607,625,903 |             31,980,220,384
   8 |        5,607,625,903 |             37,587,846,287
   9 |        5,607,625,903 |             43,195,472,189
  10 |        5,607,625,903 |             48,803,098,092
  11 |        5,607